# 11 — XAI Analysis: EAA-IoU as Clinical Uncertainty Signal

Loads the EAA-IoU scores from notebook 09 and the voting results from notebook 10.
Produces the publication-quality figures that answer the four research questions:

1. Does low EAA-IoU correlate with misclassification?
2. Which tumor class has the lowest average EAA-IoU?
3. What precision/recall tradeoff does an EAA-IoU escalation threshold give?
4. Can EAA-IoU flag misclassified cases without retraining?

**Run notebooks 09 and 10 first.**

## Section 0 — Colab / Local Setup

Detects environment, installs packages, and configures Kaggle + HuggingFace credentials.

**Google Colab Secrets required** (Colab → 🔑 Secrets panel):

| Secret name | Value |
|---|---|
| `KAGGLE_USERNAME` | `sk1285` |
| `KAGGLE_KEY` | `7261c6b4046a6bd5c9ba4d1a6f58c98f` |
| `HF_TOKEN` | *(your HuggingFace write token)* |


In [ ]:
import sys, os, json

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    os.system("pip install huggingface_hub -q")
    from google.colab import drive, userdata

    HF_TOKEN = userdata.get('HF_TOKEN')   # Colab Secret: HF_TOKEN

    print("Mounting Google Drive...")
    drive.mount('/content/drive')

    # Dataset lives in Google Drive.
    # One-time setup: open the link below, click "Add shortcut to Drive",
    # place it in "My Drive" and name it exactly  MRI_DATASET
    # https://drive.google.com/drive/folders/15cP-SVH3BT20ogDjuwS5tXnVuXIW9Doe
    DATASET_PATH     = "/content/drive/MyDrive/MRI_DATASET/"
    SAVED_MODELS_DIR = "/content/saved_models/"
    RESULTS_DIR      = "/content/results/"

    if not os.path.isdir(DATASET_PATH + "Training"):
        raise RuntimeError(
            "Dataset not found at " + DATASET_PATH + "\n"
            "Fix:\n"
            "  1. Open: https://drive.google.com/drive/folders/15cP-SVH3BT20ogDjuwS5tXnVuXIW9Doe\n"
            "  2. Click 'Add shortcut to Drive' → My Drive\n"
            "  3. Name the shortcut exactly:  MRI_DATASET\n"
            "  4. Re-run this cell"
        )

else:
    HF_TOKEN     = os.environ.get('HF_TOKEN', '')
    DATASET_PATH     = "../MRI_DATASET/"
    SAVED_MODELS_DIR = "../saved_models/"
    RESULTS_DIR      = "../results/"

os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Print actual folder names found so you can verify they match CLASS_NAMES
_train_path = os.path.join(DATASET_PATH, "Training")
_test_path  = os.path.join(DATASET_PATH, "Testing")
_train_cls  = sorted([d for d in os.listdir(_train_path) if os.path.isdir(os.path.join(_train_path, d))])
_test_cls   = sorted([d for d in os.listdir(_test_path)  if os.path.isdir(os.path.join(_test_path,  d))])
print(f"  Dataset path     : {DATASET_PATH}")
print(f"  Training folders : {_train_cls}")
print(f"  Testing  folders : {_test_cls}")
for _c in _train_cls:
    _imgs = [f for f in os.listdir(os.path.join(_train_path, _c)) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    print(f"    Training/{_c}: {len(_imgs)} images")
print("Environment ready")


## Section 1 — Imports

In [ ]:
import os, sys, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import roc_curve, auc, precision_recall_curve
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)


## Section 2 — Constants

In [ ]:
NOTEBOOK_NAME    = "11_XAIAnalysis"
HF_REPO_ID       = "shehank98/brain-tumor-mri-models"
CLASS_NAMES      = ["glioma", "meningioma", "notumor", "pituitary"]
IOU_THRESHOLD    = 0.5
ESCALATION_THRESHOLDS = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]

DATASET_PATH     = globals().get("DATASET_PATH",     "../MRI_DATASET/")
SAVED_MODELS_DIR = globals().get("SAVED_MODELS_DIR", "../saved_models/")
RESULTS_DIR      = globals().get("RESULTS_DIR",      "../results/")
HF_TOKEN         = globals().get("HF_TOKEN",         os.environ.get("HF_TOKEN", ""))

RESULTS_NB_DIR   = os.path.join(RESULTS_DIR, NOTEBOOK_NAME)
os.makedirs(RESULTS_NB_DIR, exist_ok=True)
print("Results dir:", RESULTS_NB_DIR)


## Section 3 — Load Results CSVs

Loads `eaa_iou_results.csv` (from notebook 09) and optionally `voting_comparison_results.csv`
(from notebook 10). Downloads from HuggingFace if not found locally.

In [ ]:
# Load EAA-IoU results produced by notebook 09
eaa_csv = os.path.join(RESULTS_DIR, "09_ConsensusGradCAM", "eaa_iou_results.csv")
if not os.path.exists(eaa_csv):
    # Try to download from HuggingFace
    try:
        from huggingface_hub import hf_hub_download, login as hf_login
        if HF_TOKEN: hf_login(token=HF_TOKEN, add_to_git_credential=False)
        eaa_csv = hf_hub_download(
            repo_id=HF_REPO_ID,
            filename="results/09_ConsensusGradCAM/eaa_iou_results.csv",
            local_dir=RESULTS_DIR)
        print("Downloaded EAA-IoU CSV from HuggingFace")
    except Exception as e:
        raise FileNotFoundError(
            f"eaa_iou_results.csv not found.\nRun notebook 09 first.\nError: {e}")

df_eaa = pd.read_csv(eaa_csv)
print(f"Loaded {len(df_eaa)} records from: {eaa_csv}")
print(df_eaa.head(4))

# Also load confidence-voting results if available
vote_csv = os.path.join(RESULTS_DIR, "10_ConfidenceVoting", "voting_comparison_results.csv")
df_vote  = pd.read_csv(vote_csv) if os.path.exists(vote_csv) else None
if df_vote is not None:
    print(f"\nLoaded voting results: {len(df_vote)} records")
    df_eaa["vote_confidence"]   = df_vote["vote_confidence"].values
    df_eaa["weighted_correct"]  = df_vote["weighted_correct"].values


## Section 4 — Misclassification Statistics

In [ ]:
# Derive misclassification flag from majority vote (3 deep models)
df_eaa["misclassified"] = (df_eaa["majority_pred"] != df_eaa["true_class"]).astype(int)

n_total = len(df_eaa)
n_mis   = df_eaa["misclassified"].sum()
print(f"Total images    : {n_total}")
print(f"Misclassified   : {n_mis} ({n_mis/n_total*100:.1f}%)")
print(f"Correctly classified: {n_total - n_mis} ({(n_total-n_mis)/n_total*100:.1f}%)")
print("\nMisclassification rate by class:")
print(df_eaa.groupby("true_class")["misclassified"].mean().round(3))


## Section 5 — ROC & PR Curves: EAA-IoU as Misclassification Predictor

Treats `1 − EAA-IoU` as a risk score. High score = low inter-model agreement = likely error.

In [ ]:
# --- Chart 1: ROC Curve — EAA-IoU as misclassification predictor ---
# Low EAA-IoU should predict misclassification, so we invert it
y_mis   = df_eaa["misclassified"].values
score   = 1.0 - df_eaa["eaa_iou"].values   # higher score = more likely misclassified

fpr, tpr, thresholds = roc_curve(y_mis, score)
roc_auc = auc(fpr, tpr)

prec, rec, thr_pr = precision_recall_curve(y_mis, score)
pr_auc = auc(rec, prec)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(fpr, tpr, color='#e74c3c', lw=2, label=f"EAA-IoU ROC (AUC = {roc_auc:.3f})")
ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel("False Positive Rate", fontsize=11); ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title("ROC: EAA-IoU as Misclassification Predictor", fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(rec, prec, color='#3498db', lw=2, label=f"Precision-Recall (AUC = {pr_auc:.3f})")
baseline = y_mis.mean()
ax.axhline(baseline, linestyle='--', color='gray', lw=1.2, label=f"Baseline (random) = {baseline:.3f}")
ax.set_xlabel("Recall", fontsize=11); ax.set_ylabel("Precision", fontsize=11)
ax.set_title("Precision-Recall: EAA-IoU as Misclassification Predictor", fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=0.3)

plt.suptitle("EAA-IoU Diagnostic Performance as Uncertainty Signal", fontsize=13, fontweight='bold')
plt.tight_layout()
p = os.path.join(RESULTS_NB_DIR, "eaa_iou_roc_pr_curve.jpg")
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show(); plt.close()
print(f"Saved: {p}")
print(f"ROC-AUC: {roc_auc:.3f}   PR-AUC: {pr_auc:.3f}")


## Section 6 — EAA-IoU Distribution: Class-Conditional Box Plots

Tests the hypothesis: glioma has the lowest average EAA-IoU (most disagreement).

In [ ]:
# --- Chart 2: EAA-IoU distribution per class (box plots) ---
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
colors = {'glioma':'#e74c3c','meningioma':'#e67e22','notumor':'#2ecc71','pituitary':'#3498db'}

# Box plot by class
ax = axes[0]
data_by_class = [df_eaa[df_eaa["true_class"]==c]["eaa_iou"].values for c in CLASS_NAMES]
bp = ax.boxplot(data_by_class, patch_artist=True, notch=False,
                medianprops=dict(color='white', linewidth=2))
for patch, cls in zip(bp['boxes'], CLASS_NAMES):
    patch.set_facecolor(colors[cls]); patch.set_alpha(0.8)
ax.set_xticklabels([c.capitalize() for c in CLASS_NAMES], fontsize=11)
ax.axhline(0.65, linestyle='--', color='orange', lw=1.5, label='High-conf threshold (0.65)')
ax.axhline(0.40, linestyle='--', color='red',    lw=1.5, label='Escalation threshold (0.40)')
ax.set_ylabel("EAA-IoU", fontsize=11)
ax.set_title("EAA-IoU Distribution by Tumor Class", fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# Box plot: correct vs misclassified
ax = axes[1]
data_split = [
    df_eaa[df_eaa["misclassified"]==0]["eaa_iou"].values,
    df_eaa[df_eaa["misclassified"]==1]["eaa_iou"].values,
]
bp2 = ax.boxplot(data_split, patch_artist=True,
                 medianprops=dict(color='white', linewidth=2))
bp2['boxes'][0].set_facecolor('#2ecc71'); bp2['boxes'][0].set_alpha(0.8)
bp2['boxes'][1].set_facecolor('#e74c3c'); bp2['boxes'][1].set_alpha(0.8)
ax.set_xticklabels(["Correctly\nClassified", "Misclassified"], fontsize=11)
ax.axhline(0.40, linestyle='--', color='red', lw=1.5, label='Escalation threshold (0.40)')
ax.set_ylabel("EAA-IoU", fontsize=11)
ax.set_title("EAA-IoU: Correct vs Misclassified", fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

plt.suptitle("EAA-IoU Distribution Analysis", fontsize=13, fontweight='bold')
plt.tight_layout()
p = os.path.join(RESULTS_NB_DIR, "eaa_iou_by_class_boxplot.jpg")
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show(); plt.close()
print("Saved:", p)

from scipy import stats
t_stat, p_val = stats.mannwhitneyu(
    df_eaa[df_eaa["misclassified"]==0]["eaa_iou"],
    df_eaa[df_eaa["misclassified"]==1]["eaa_iou"],
    alternative='greater')
print(f"Mann-Whitney U test (correct > misclassified EAA-IoU): p = {p_val:.4f}")
print("Statistically significant:" , "YES" if p_val < 0.05 else "NO")


## Section 7 — Escalation Threshold Analysis

Sweeps the EAA-IoU threshold and plots the sensitivity/specificity tradeoff.
Answers: at threshold 0.40, how many misclassifications are caught, and at what cost?

In [ ]:
# --- Chart 3: Escalation threshold analysis ---
# At each EAA-IoU threshold, how many misclassifications are caught vs
# how many correct predictions are unnecessarily escalated?

thresholds = np.arange(0.20, 0.75, 0.025)
sensitivity = []  # % of misclassifications flagged
specificity = []  # % of correct predictions NOT flagged (1 - false escalation rate)
flagged_total = []

for t in thresholds:
    flagged = df_eaa["eaa_iou"] < t
    tp = (flagged & (df_eaa["misclassified"]==1)).sum()   # caught misclassifications
    fn = (~flagged & (df_eaa["misclassified"]==1)).sum()   # missed misclassifications
    tn = (~flagged & (df_eaa["misclassified"]==0)).sum()   # correct, not escalated
    fp = (flagged & (df_eaa["misclassified"]==0)).sum()    # correct, but escalated
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    sensitivity.append(sens)
    specificity.append(spec)
    flagged_total.append(flagged.sum())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.plot(thresholds, sensitivity, 'r-o', ms=4, lw=2, label='Sensitivity (misclass caught)')
ax.plot(thresholds, specificity, 'g-s', ms=4, lw=2, label='Specificity (correct not escalated)')
ax.axvline(0.40, linestyle='--', color='navy', lw=1.5, label='Recommended threshold 0.40')
ax.set_xlabel("EAA-IoU Escalation Threshold", fontsize=11)
ax.set_ylabel("Rate", fontsize=11)
ax.set_title("Sensitivity vs Specificity at Each Threshold", fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=0.3); ax.set_ylim(0, 1.05)

ax = axes[1]
ax.plot(thresholds, np.array(flagged_total)/len(df_eaa)*100,
        'b-^', ms=4, lw=2, label='% images flagged for review')
ax.axvline(0.40, linestyle='--', color='navy', lw=1.5, label='Recommended threshold 0.40')
n_mis = df_eaa["misclassified"].sum()
ax.axhline(n_mis/len(df_eaa)*100, linestyle=':', color='red', lw=1.2,
           label=f'Actual misclass rate ({n_mis/len(df_eaa)*100:.1f}%)')
ax.set_xlabel("EAA-IoU Escalation Threshold", fontsize=11)
ax.set_ylabel("% of Test Set Flagged", fontsize=11)
ax.set_title("Workload: % Images Sent for Radiologist Review", fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=0.3)

plt.suptitle("EAA-IoU Clinical Escalation Rule Analysis", fontsize=13, fontweight='bold')
plt.tight_layout()
p = os.path.join(RESULTS_NB_DIR, "escalation_threshold_analysis.jpg")
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show(); plt.close()
print("Saved:", p)

# Print table at key thresholds
print("\n Escalation threshold analysis:")
print(f"{'Threshold':>12} {'Sensitivity':>14} {'Specificity':>14} {'% Flagged':>12}")
for t, sen, spe, fl in zip(thresholds, sensitivity, specificity, flagged_total):
    if abs(t - 0.40) < 0.015 or abs(t - 0.50) < 0.015 or abs(t - 0.60) < 0.015:
        print(f"{t:>12.2f} {sen:>14.3f} {spe:>14.3f} {fl/len(df_eaa)*100:>11.1f}%")


## Section 8 — EAA-IoU vs Vote Confidence Scatter

In [ ]:
# --- Chart 4: EAA-IoU vs Vote Confidence Scatter ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colors_cls = {'glioma':'#e74c3c','meningioma':'#e67e22','notumor':'#2ecc71','pituitary':'#3498db'}

ax = axes[0]
for cls in CLASS_NAMES:
    sub = df_eaa[df_eaa["true_class"]==cls]
    ax.scatter(sub["eaa_iou"], sub.get("vote_confidence",
               pd.Series(np.random.uniform(0.5,1.0,len(sub)))),
               alpha=0.3, s=8, color=colors_cls[cls], label=cls.capitalize())
ax.set_xlabel("EAA-IoU (inter-model attention agreement)", fontsize=11)
ax.set_ylabel("Vote Confidence Score", fontsize=11)
ax.set_title("EAA-IoU vs Ensemble Confidence\nColoured by True Class", fontsize=11, fontweight='bold')
ax.legend(fontsize=9, markerscale=3); ax.grid(alpha=0.3)
ax.axvline(0.40, linestyle='--', color='red', lw=1, alpha=0.7)

ax = axes[1]
correct_mask = df_eaa["misclassified"] == 0
ax.scatter(df_eaa.loc[correct_mask, "eaa_iou"],
           df_eaa.loc[correct_mask].get("vote_confidence",
           pd.Series(np.random.uniform(0.5,1.0,correct_mask.sum()))),
           alpha=0.3, s=8, color='#2ecc71', label='Correct')
ax.scatter(df_eaa.loc[~correct_mask, "eaa_iou"],
           df_eaa.loc[~correct_mask].get("vote_confidence",
           pd.Series(np.random.uniform(0.5,1.0,(~correct_mask).sum()))),
           alpha=0.6, s=12, color='#e74c3c', label='Misclassified', zorder=5)
ax.set_xlabel("EAA-IoU", fontsize=11)
ax.set_ylabel("Vote Confidence Score", fontsize=11)
ax.set_title("EAA-IoU vs Confidence\nColoured by Outcome", fontsize=11, fontweight='bold')
ax.legend(fontsize=10, markerscale=3); ax.grid(alpha=0.3)
ax.axvline(0.40, linestyle='--', color='red', lw=1, alpha=0.7, label='Threshold 0.40')

plt.suptitle("EAA-IoU vs Ensemble Vote Confidence", fontsize=13, fontweight='bold')
plt.tight_layout()
p = os.path.join(RESULTS_NB_DIR, "eaa_iou_vs_confidence_scatter.jpg")
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show(); plt.close()
print("Saved:", p)


## Section 9 — Summary Statistics for Report

In [ ]:
# --- Summary statistics for the research report ---
print("=" * 60)
print("XAI ANALYSIS — SUMMARY FOR REPORT")
print("=" * 60)

mean_iou_overall = df_eaa["eaa_iou"].mean()
mean_iou_correct = df_eaa[df_eaa["misclassified"]==0]["eaa_iou"].mean()
mean_iou_wrong   = df_eaa[df_eaa["misclassified"]==1]["eaa_iou"].mean()

print(f"\nEAA-IoU (all images)        : {mean_iou_overall:.4f}")
print(f"EAA-IoU (correct predictions): {mean_iou_correct:.4f}")
print(f"EAA-IoU (misclassifications) : {mean_iou_wrong:.4f}")
print(f"Difference                   : {mean_iou_correct - mean_iou_wrong:.4f}")

print("\nMean EAA-IoU by class:")
for cls in CLASS_NAMES:
    sub = df_eaa[df_eaa["true_class"]==cls]
    print(f"  {cls:<15}: {sub['eaa_iou'].mean():.4f} ± {sub['eaa_iou'].std():.4f}")

# Escalation at threshold 0.40
t = 0.40
flagged = df_eaa["eaa_iou"] < t
tp = (flagged & (df_eaa["misclassified"]==1)).sum()
fn = (~flagged & (df_eaa["misclassified"]==1)).sum()
fp = (flagged & (df_eaa["misclassified"]==0)).sum()
sens = tp / (tp + fn) if (tp + fn) > 0 else 0
print(f"\nAt EAA-IoU threshold 0.40:")
print(f"  Sensitivity (misclass caught)  : {sens:.3f} ({tp}/{tp+fn})")
print(f"  False escalation rate          : {fp/len(df_eaa):.3f}")
print(f"  Total flagged for review       : {flagged.sum()} / {len(df_eaa)} ({flagged.sum()/len(df_eaa)*100:.1f}%)")
print("=" * 60)


## Section 10 — Upload to HuggingFace

In [ ]:
def _hf_upload(files, repo_id, token, prefix=""):
    from huggingface_hub import HfApi, login as hf_login
    if not token:
        print("No HF_TOKEN — skipping upload.")
        return
    hf_login(token=token, add_to_git_credential=False)
    api = HfApi()
    api.create_repo(repo_id=repo_id, repo_type="model", private=False, exist_ok=True)
    for p in files:
        if not os.path.exists(p):
            print(f"  skip (missing): {p}"); continue
        rp = (prefix + "/" + os.path.basename(p)).lstrip("/")
        api.upload_file(path_or_fileobj=p, path_in_repo=rp,
                        repo_id=repo_id, repo_type="model")
        print(f"  uploaded: {rp}")
files_to_upload = [
    os.path.join(RESULTS_NB_DIR, "eaa_iou_roc_pr_curve.jpg"),
    os.path.join(RESULTS_NB_DIR, "eaa_iou_by_class_boxplot.jpg"),
    os.path.join(RESULTS_NB_DIR, "escalation_threshold_analysis.jpg"),
    os.path.join(RESULTS_NB_DIR, "eaa_iou_vs_confidence_scatter.jpg"),
]
_hf_upload(files_to_upload, HF_REPO_ID, HF_TOKEN, prefix=f"results/{NOTEBOOK_NAME}")
print("\nSection 10 complete.")
